In [ ]:
import pandas as pd

df = pd.read_csv("sample_data/sp500_panel.csv")
df.head()

In [ ]:
df.shape
df.info()
df.describe()

In [ ]:
df["Ticker"].nunique()
df["Date"].min(), df["Date"].max()
df.groupby("Ticker").size().describe()

In [ ]:
import matplotlib.pyplot as plt

df["Close"].hist(bins=50)
plt.title("Distribution of Closing Prices")
plt.show()

In [ ]:
#normal garph was not too ueful so log distrbution for better understandig
import numpy as np

np.log(df["Close"]).hist(bins=50)
plt.title("Log Distribution of Closing Prices")
plt.show()

#appears as a gaussian curve

In [ ]:
df["Volume"].describe()
df["Volume"].hist(bins=50)
plt.title("Trading Volume Distribution")
plt.show()
top_volume = df.groupby("Ticker")["Volume"].mean().sort_values(ascending=False).head(10)
top_volume.plot(kind="bar", title="Top 10 Stocks by Avg Volume")
plt.show()

#10 best perfroming stocks , NVidia , Tesla .....

In [ ]:
#avg prices over the time duration data covers
market_avg = df.groupby("Date")["Close"].mean()

market_avg.plot(figsize=(10,5), title="Average S&P500 Stock Price Over Time")
plt.show()
#increasing trend observed

In [ ]:
tickers = ["NVDA","TSLA","AAPL", "MSFT", "GOOGL", "AMZN"]

subset = df[df["Ticker"].isin(tickers)]

for t in tickers:
    subset[subset["Ticker"] == t].set_index("Date")["Close"].plot(label=t, figsize=(10,5))

plt.legend()
plt.title("Stock Price Trends")
plt.show()

Overall Observations So far:

1. Basic Info about data -:

total data entries : 1218220

columns: 16

data types: float64(12), int64(2), object(2)

2. Important data analysis values:

count 501.000000

mean 2431.576846

std 336.325100

min 34.000000

25% 2510.000000

50% 2510.000000

75% 2510.000000

max 2510.000000

// Also a quick check to make sure null values were removed during pre-processing.

3. Graphical Distributions:

Normal closing graph did not help much, also made a log graph, appears as a gaussian curve.
4. Top 10 countries:

Nvidia, Tesla, Apple, Netflix, Amazon, AMD, BAC, PLTR, F, T

5. Avg & Individual stocks

 Average stock trends for all companies an few individual companies.

 a. Nvidia had a flat growth in the initial years and started picking up in 2023.

 b. Tesla stocks started slowly piking in 2019 but keep rigourously fluctuating between pikes and lows.

 c. Other stocks like Apple, Microsoft, Google and Amazon all started picking up in 2019 and are still rising with microsoft having the hightest peak in 2025.

 d. General observation is that except Nvidia, every other company started peaking in 2019 but NVidia was abit later in 2021.  


Lakku's part:

In [ ]:
# 3) Peek at columns only (without loading entire file)

import pandas as pd

sample = pd.read_csv(drive_path, nrows=5)
print("Columns:", list(sample.columns))
display(sample)


# 4) Memory-safe full row count in chunks
chunksize = 300_000   # increase/decrease based on RAM
total_rows = 0

for chunk in pd.read_csv(drive_path, chunksize=chunksize):
    total_rows += len(chunk)

print("Total rows:", total_rows)


# 5) OPTIONAL: Basic chunked summary (edit column names if needed)
# This block assumes columns named: 'ticker', 'close'
# If your column names differ, update usecols and references below.

sum_close = 0.0
count_close = 0
ticker_counts = {}

for chunk in pd.read_csv(drive_path, chunksize=chunksize, usecols=['ticker', 'close']):
    close = pd.to_numeric(chunk['close'], errors='coerce')
    sum_close += close.fillna(0).sum()
    count_close += close.notna().sum()

    vc = chunk['ticker'].value_counts()
    for t, c in vc.items():
        ticker_counts[t] = ticker_counts.get(t, 0) + int(c)

mean_close = (sum_close / count_close) if count_close else None
print("Mean close:", mean_close)
print("Unique tickers:", len(ticker_counts))


# 6) OPTIONAL: Convert to Parquet once (much faster next time)
# Requires pyarrow (usually available in Colab, install if needed)
# !pip -q install pyarrow

parquet_path = drive_path.replace(".csv", ".parquet")
first = True

for chunk in pd.read_csv(drive_path, chunksize=chunksize):
    # append-like write by creating partitioned parts
    part_path = parquet_path.replace(".parquet", f"_part_{'0000' if first else str(total_rows)}.parquet")
    chunk.to_parquet(part_path, index=False)
    first = False

print("Parquet parts written in:", drive_folder)
